In [1]:
from pyspark.sql import SparkSession
from pathlib import Path

In [2]:
jar_filename = "sqlite-jdbc-3.53.0.0.jar"
jar_path = Path(jar_filename)
jar_path.exists()

True

In [3]:
str(jar_path.resolve())

'/home/jovyan/work/sqlite-jdbc-3.53.0.0.jar'

In [ ]:
spark = (
    SparkSession.builder
    .appName("Northwind-OLAP-Spark")
    .config("spark.jars", str(jar_path.resolve()))
    # Desativa a Web UI para economizar memória e evitar loops de porta
    # .config("spark.ui.enabled", "false")
    # Força o modo local com apenas 1 thread para teste inicial
    # .master("local[1]")
    # Resolve o problema de Hostname do Docker
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    # Garante que a JVM não tente alocar mais do que o container permite
    .config("spark.driver.memory", "512m")
    .getOrCreate()
)

print("Spark Session iniciada com sucesso!")

In [ ]:
df_fact = (
    spark.read.format("jdbc")
    .option("url", "jdbc:sqlite:/home/jovyan/work/northwind-dw.db")
    .option("dbtable", "fact_sales")
    .load()
)

In [ ]:
df_fact.show(5)

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Path and existence check
jar_path = "/home/jovyan/work/sqlite-jdbc-3.53.0.0.jar" # Rename your file to this
if not os.path.exists(jar_path):
    print(f"CRITICAL: JAR not found at {jar_path}")

# 2. Force Python to use the internal Docker IP
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .appName("Northwind-OLAP-Spark")
    .master("local[1]") # Use only 1 core to stabilize
    .config("spark.jars", jar_path)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    # Low memory allocation to avoid JVM overhead crashes
    .config("spark.driver.memory", "1g")
    .config("spark.sql.shuffle.partitions", "1")
    # Set log level to see what is happening
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("SUCCESS: Spark is alive.")

In [ ]:
# --- CARGA DOS DADOS BRUTOS ---
# Simulando a leitura das tabelas do SQLite para Spark DataFrames
df_orders = spark.read.format("jdbc").options(url="jdbc:sqlite:northwind.db", dbtable="Orders").load()
df_details = spark.read.format("jdbc").options(url="jdbc:sqlite:northwind.db", dbtable="Order Details").load()
df_products = spark.read.format("jdbc").options(url="jdbc:sqlite:northwind.db", dbtable="Products").load()

# --- TRANSFORMAÇÃO: DIM_TIME ---
# Em vez de um loop while, usamos funções de data do Spark
df_dim_time = df_orders.select("OrderDate").distinct() \
    .withColumn("dt", F.date_format("OrderDate", "yyyy-MM-dd HH")) \
    .withColumn("year", F.year("OrderDate")) \
    .withColumn("month", F.month("OrderDate")) \
    .withColumn("quarter", F.quarter("OrderDate")) \
    .withColumn("day_period", 
        F.when((F.hour("OrderDate") >= 6) & (F.hour("OrderDate") <= 11), "Morning")
         .when((F.hour("OrderDate") >= 12) & (F.hour("OrderDate") <= 17), "Afternoon")
         .otherwise("Night"))
# Nota: O ID seria gerado com F.monotonically_increasing_id()

# --- TRANSFORMAÇÃO: FACT_SALES ---
# Unindo tabelas para criar a Fato
df_fact = df_orders.join(df_details, "OrderID") \
    .withColumn("revenue_net", F.round((F.col("UnitPrice") * F.col("Quantity")) * (1 - F.col("Discount")), 2)) \
    .select(
        F.col("OrderID").alias("order_id"),
        F.col("OrderDate").alias("order_dt"),
        "CustomerID", "ProductID", "EmployeeID", "UnitPrice", "Quantity", "Discount", "revenue_net"
    )

In [ ]:
# Substituindo a query SQL do DuckDB por API de DataFrame
result = df_fact.join(df_dim_time, df_fact.order_dt == df_dim_time.OrderDate) \
    .rollup("year", "quarter", "month") \
    .agg(F.count("order_id").alias("total_orders")) \
    .orderBy(
        F.asc_nulls_last("year"),
        F.asc_nulls_last("quarter"),
        F.asc_nulls_last("month")
    )

# Exibe o resultado (equivalente ao tabular())
result.show(n=100)